# PINN(Physics-Informed Neural Networks)을 활용한 편미분 방정식 해결

물리 정보 신경망(PINN)은 데이터뿐만 아니라 물리 법칙(편미분 방정식, PDE)을 손실 함수(Loss Function)에 포함하여 학습하는 딥러닝 기법입니다.
이 튜토리얼에서는 기존의 다층 퍼셉트론(MLP) 기반 튜토리얼을 확장하여, PINN을 사용해 2차원 공간에서의 **라플라스 방정식(Laplace Equation)**과 **푸아송 방정식(Poisson Equation)**을 해결하는 과정을 단계별로 알아봅니다.

## 1. 입력 데이터 설명 (도메인 및 경계 조건)

PINN은 두 가지 유형의 입력 데이터 포인트가 필요합니다:
1. **$X_u$ (경계 조건 포인트)**: 도메인의 경계선에 위치하며, 우리가 이미 정답(경계 조건)을 알고 있는 점들입니다.
2. **$X_f$ (내부 물리 포인트)**: 도메인 내부에 위치하며, 정답은 모르지만 물리 법칙(편미분 방정식)을 만족해야 하는 점들입니다.

여기서는 $(x, y) \in [0, 1] \times [0, 1]$ 인 2차원 정사각형 도메인을 정의합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import time

# 재현성을 위한 난수 시드 고정
torch.manual_seed(42)
np.random.seed(42)

# ==========================================
# 1. 도메인 및 데이터 생성
# ==========================================

N_u = 100   # 경계 조건(Boundary Condition) 포인트 수
N_f = 2000  # 내부 물리 검증(PDE) 포인트 수

# 1-1. 경계 조건 포인트 생성 (x=0, x=1, y=0, y=1)
pts_per_edge = N_u // 4
x_bc_0 = torch.zeros(pts_per_edge, 1)
y_bc_0 = torch.rand(pts_per_edge, 1) # x=0 경계

x_bc_1 = torch.ones(pts_per_edge, 1)
y_bc_1 = torch.rand(pts_per_edge, 1) # x=1 경계

x_bc_2 = torch.rand(pts_per_edge, 1)
y_bc_2 = torch.zeros(pts_per_edge, 1) # y=0 경계

x_bc_3 = torch.rand(pts_per_edge, 1)
y_bc_3 = torch.ones(pts_per_edge, 1) # y=1 경계

X_u_x = torch.cat([x_bc_0, x_bc_1, x_bc_2, x_bc_3])
X_u_y = torch.cat([y_bc_0, y_bc_1, y_bc_2, y_bc_3])

X_u = torch.cat([X_u_x, X_u_y], dim=1)

# 1-2. 내부 물리 검증 포인트 생성 (균등 분포 샘플링)
X_f_x = torch.rand(N_f, 1)
X_f_y = torch.rand(N_f, 1)
X_f = torch.cat([X_f_x, X_f_y], dim=1)

# PINN 학습을 위해 X_f는 미분 대상이므로 requires_grad=True 설정이 필수입니다.
X_f.requires_grad_(True)

print(f"경계 조건 포인트 X_u 형태: {X_u.shape}")
print(f"내부 물리 포인트 X_f 형태: {X_f.shape}")

# 도메인 포인트 시각화
plt.figure(figsize=(6,6))
plt.scatter(X_f[:,0].detach(), X_f[:,1].detach(), c='lightgray', s=10, label='Interior Points (X_f)')
plt.scatter(X_u[:,0].detach(), X_u[:,1].detach(), c='red', s=20, label='Boundary Points (X_u)')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Collocation Points for PINN')
plt.legend(loc='upper right')
plt.xlim(-0.05, 1.05)
plt.ylim(-0.05, 1.05)
plt.grid(True, alpha=0.3)
plt.show()


## 2. 하이퍼파라미터 설정 및 신경망 구조 시각화

PINN은 물리 법칙이라는 복잡한 규제를 만족해야 하므로 깊은 신경망 구조가 필요합니다.
사용자의 설정에 따라 **5개의 은닉층(Hidden Layer)과 각 층당 64개의 노드**를 가지도록 구성합니다.
또한, PINN에서 2차 편미분($u_{xx}, u_{yy}$)을 매끄럽게 계산하기 위해, 기울기가 일정한 `ReLU` 대신 연속적으로 미분이 가능한 **`Tanh`**를 활성화 함수로 사용합니다. 출력층은 어떠한 제약 없이 실수값을 예측하도록 선형(Linear) 형태로 둡니다.

In [ ]:
# ==========================================
## 2. 하이퍼파라미터 설정
# ==========================================

learning_rate = 1e-3 # PINN 학습에 유리한 Adam의 기본 학습률
epochs = 5000
# 은닉층 노드 수 리스트 (5개의 은닉층, 64노드)
hidden_layers = [64, 64, 64, 64, 64]
# PINN을 위한 활성화 함수 설정
activation_func = nn.Tanh()

print(f"학습률: {learning_rate}")
print(f"목표 에포크: {epochs}")
print(f"은닉층 구조: {hidden_layers}")
print(f"활성화 함수: {activation_func.__class__.__name__}")

def draw_neural_net(input_size, hidden_layers, output_size):
    """
    설정된 파라미터에 따라 PINN 신경망 구조를 그립니다.
    노드가 64개로 많아 화면에 다 그릴 수 없으므로, 일부 노드만 그리고 축약 기호를 넣습니다.
    """
    # 시각화를 위해 노드 표시를 층당 최대 8개로 제한
    vis_hidden = [min(h, 8) for h in hidden_layers]
    layer_sizes = [input_size] + vis_hidden + [output_size]
    
    fig = plt.figure(figsize=(10, 6))
    ax = fig.gca()
    ax.axis('off')
    
    left, right, bottom, top = 0.1, 0.9, 0.1, 0.9
    v_spacing = (top - bottom) / float(max(layer_sizes))
    h_spacing = (right - left) / float(len(layer_sizes) - 1)
    
    # 노드 그리기
    for n, layer_size in enumerate(layer_sizes):
        layer_top = v_spacing * (layer_size - 1) / 2. + (top + bottom) / 2.
        for m in range(layer_size):
            circle = plt.Circle((n * h_spacing + left, layer_top - m * v_spacing), v_spacing / 5.,
                                color='w', ec='b', zorder=4)
            ax.add_artist(circle)
            # 중간 노드 생략(점선) 표시
            if m == layer_size - 1 and hidden_layers[0] > 8 and n > 0 and n < len(layer_sizes)-1:
                ax.text(n * h_spacing + left, layer_top - (m+1.5) * v_spacing, '\n...\n(64)', ha='center', va='center', fontsize=10)
            
    # 간선 그리기
    for n, (layer_size_a, layer_size_b) in enumerate(zip(layer_sizes[:-1], layer_sizes[1:])):
        layer_top_a = v_spacing * (layer_size_a - 1) / 2. + (top + bottom) / 2.
        layer_top_b = v_spacing * (layer_size_b - 1) / 2. + (top + bottom) / 2.
        for m in range(layer_size_a):
            for o in range(layer_size_b):
                line = plt.Line2D([n * h_spacing + left, (n + 1) * h_spacing + left],
                                  [layer_top_a - m * v_spacing, layer_top_b - o * v_spacing], c='k', alpha=0.1)
                ax.add_artist(line)
    
    plt.title(f"PINN Architecture (Input: {input_size} -> {len(hidden_layers)} Hidden Layers of {hidden_layers[0]} Nodes -> Output: {output_size})")
    plt.show()

draw_neural_net(2, hidden_layers, 1)


## 3. 다이내믹 모델 생성 및 첫 순전파(Forward Pass) 시각화

설정한 5층, 64노드의 구조에 맞춰 딥러닝 모델을 동적으로 생성합니다.  
PINN 모델 $u_{\theta}(x,y)$ 는 위치 $(x,y)$ 를 입력으로 받아 해당 위치에서의 해 $u$ 를 예측하는 함수로 동작합니다.
초기화된 가중치를 사용하여 내부 물리 포인트(`X_f`) 일부에 대해 순전파를 진행해보고, 은닉층을 통과하며 데이터가 어떻게 변하는지 히트맵으로 관찰합니다.

In [ ]:
# ==========================================
## 3. 다이내믹 모델 생성 및 첫 순전파
# ==========================================

class DynamicPINN(nn.Module):
    def __init__(self, input_size, hidden_layers, output_size, activation):
        super(DynamicPINN, self).__init__()
        self.layers = nn.ModuleList()
        self.activation = activation
        
        # 은닉층 생성
        in_size = input_size
        for h_size in hidden_layers:
            self.layers.append(nn.Linear(in_size, h_size))
            in_size = h_size
            
        # 출력층 생성 (제약 없는 선형 출력)
        self.output_layer = nn.Linear(in_size, output_size)
        
    def forward(self, x):
        activations = [] # 각 레이어의 출력값을 저장
        out = x
        for layer in self.layers:
            out = layer(out)
            out = self.activation(out)
            activations.append(out)
        out = self.output_layer(out)
        activations.append(out)
        return out, activations

# 모델 인스턴스화
model = DynamicPINN(input_size=2, hidden_layers=hidden_layers, output_size=1, activation=activation_func)

# 첫 순전파 수행 (시각화를 위해 X_f 중 10개의 데이터 샘플만 사용)
sample_X = X_f[:10]
predictions, activations = model(sample_X)

print("[첫 순전파 결과 (상위 10개 샘플)]")
print("입력 좌표 X (x, y):\n", sample_X.detach().numpy()[:3], "...\n")
print("예측 결과 u(x, y):\n", predictions.detach().numpy()[:3], "...")

# 시각화 (히트맵 - 5개의 은닉층과 1개의 출력층)
fig, axes = plt.subplots(1, len(activations), figsize=(4 * len(activations), 4))
for i, (act, ax) in enumerate(zip(activations, axes)):
    # 노드가 64개이므로 값도 많습니다. matshow로 밀도 있게 표현
    cax = ax.matshow(act.detach().numpy(), cmap='viridis', aspect='auto')
    title = f"Hidden Layer {i+1} Output" if i < len(activations)-1 else "Final Output"
    ax.set_title(title)
    ax.set_xlabel("Nodes" if i < len(activations)-1 else "Output")
    if i == 0: ax.set_ylabel("Data Index (10 samples)")
    fig.colorbar(cax, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


## 4. 오차(Loss) 계산 과정 시각화 (PINN의 핵심)

딥러닝에서 모델을 학습시키는 기준이 되는 오차(Loss)를 계산합니다.
PINN의 총 오차는 다음 두 가지 요소의 합으로 이루어집니다:

$$Total\ Loss = MSE_u + MSE_f$$

1. **$MSE_u$ (경계 조건 오차)**: 도메인의 가장자리($X_u$)에서 모델 예측값이 정해진 정답을 얼마나 잘 맞추는지 계산합니다.
2. **$MSE_f$ (물리 법칙 오차)**: 도메인 내부($X_f$)에서 모델 예측값이 주어진 물리 미분방정식을 얼마나 잘 따르는지 계산합니다. 라플라스 방정식의 경우 $\Delta u = u_{xx} + u_{yy} = 0$ 이 성립해야 합니다.

PyTorch의 강력한 기능인 **`torch.autograd.grad` (자동 미분)** 기능을 사용해 공간에 대한 모델의 2차 편미분 값을 구하는 과정을 살펴봅니다.

In [ ]:
# ==========================================
## 4. 오차(Loss) 계산 과정 및 편미분 시각화
# ==========================================

# 미분 과정을 보여주기 위해 샘플 5개만 추출
X_f_sample = X_f[:5]
u_pred_sample, _ = model(X_f_sample)

# 1. 자동 미분을 활용한 1차 편미분 (u_x, u_y 계산)
# create_graph=True 로 설정해야 한 번 더 미분(2차 미분)할 수 있는 계산 그래프가 보존됩니다.
grad_u = torch.autograd.grad(u_pred_sample, X_f_sample, grad_outputs=torch.ones_like(u_pred_sample), create_graph=True)[0]
u_x = grad_u[:, 0:1] # x에 대한 1차 편미분
u_y = grad_u[:, 1:2] # y에 대한 1차 편미분

# 2. 2차 편미분 (u_xx, u_yy 계산)
grad_u_x = torch.autograd.grad(u_x, X_f_sample, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
u_xx = grad_u_x[:, 0:1] # x에 대한 2차 편미분

grad_u_y = torch.autograd.grad(u_y, X_f_sample, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
u_yy = grad_u_y[:, 1:2] # y에 대한 2차 편미분

# 라플라스 방정식의 Residual (오차 잔차)
# 라플라스 방정식은 u_xx + u_yy = 0 을 만족해야 하므로 그 자체를 오차로 봅니다.
f_residual = u_xx + u_yy
mse_f_sample = torch.mean(f_residual ** 2)

print("--- 2차 편미분 연산 과정 추적 (첫 5개 샘플) ---")
print("1. u 예측값 (u_pred):\n", u_pred_sample.detach().numpy())
print("2. x 방향 기울기 (u_x):\n", u_x.detach().numpy())
print("3. x 방향 곡률 (u_xx):\n", u_xx.detach().numpy())
print("4. 라플라스 잔차 (u_xx + u_yy):\n", f_residual.detach().numpy())
print(f"\n물리 법칙 만족 여부 (MSE_f, 0에 가까울수록 좋음): {mse_f_sample.item():.6f}")

# 본격적인 학습에 사용할 공통 물리 함수 (라플라스용)를 정의합니다.
def calc_pde_loss_laplace(model, x_f):
    u = model(x_f)[0]
    grad_u = torch.autograd.grad(u, x_f, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x = grad_u[:, 0:1]
    u_y = grad_u[:, 1:2]
    
    grad_u_x = torch.autograd.grad(u_x, x_f, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_xx = grad_u_x[:, 0:1]
    
    grad_u_y = torch.autograd.grad(u_y, x_f, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_yy = grad_u_y[:, 1:2]
    
    residual = u_xx + u_yy
    return torch.mean(residual ** 2)


## 5. 역전파(Backpropagation) 및 가중치 업데이트 시각화

계산된 총 오차(Loss)를 거꾸로 흘려보내 각 가중치가 오차에 미치는 영향(Gradient)을 구하고, 옵티마이저를 통해 가중치를 학습합니다.
이 과정은 이전 튜토리얼(XOR)과 동일하지만, **PINN과 같이 깊고 복잡한 모델을 훈련시킬 때는 SGD 대신 최신 최적화 알고리즘인 `Adam`을 주로 사용합니다.**
학습 전후 첫 번째 은닉층 가중치의 변화를 히트맵으로 비교합니다.

In [ ]:
# ==========================================
## 5. 역전파 및 가중치 업데이트
# ==========================================

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 업데이트 전 가중치 저장 (첫 번째 은닉층 기준, 시각화를 위해 상위 10x2 노드만 추출)
old_weights = model.layers[0].weight.data[:10, :].clone().numpy()

# 임의의 테스트용 Loss 계산 (모든 경계에서 u=0을 만족하도록 임시 목표 설정)
u_pred_bc, _ = model(X_u)
target_bc_temp = torch.zeros_like(u_pred_bc)
mse_u = nn.MSELoss()(u_pred_bc, target_bc_temp)
mse_f = calc_pde_loss_laplace(model, X_f)
total_loss = mse_u + mse_f

# 역전파 수행
optimizer.zero_grad() # 기존 기울기 초기화
total_loss.backward() # 역전파를 통한 기울기 계산

# 계산된 기울기 추출 (상위 10x2 노드)
gradients = model.layers[0].weight.grad[:10, :].numpy()

# 가중치 업데이트 수행
optimizer.step()

# 업데이트 후 가중치 저장 (상위 10x2 노드)
new_weights = model.layers[0].weight.data[:10, :].numpy()

print(f"[1 에포크 학습 후의 Loss 구성표]")
print(f"MSE_u (경계 조건 오차): {mse_u.item():.6f}")
print(f"MSE_f (물리 법칙 오차): {mse_f.item():.6f}")
print(f"Total Loss:           {total_loss.item():.6f}\n")

print("[첫 번째 은닉층(Layer 1) 상위 10개 노드의 가중치 변화]")

# 시각화 (히트맵 비교)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

def plot_heatmap(ax, data, title):
    cax = ax.matshow(data, cmap='coolwarm')
    for (row, col), val in np.ndenumerate(data):
        ax.text(col, row, f"{val:.3f}", ha='center', va='center', color='black', fontsize=9)
    ax.set_title(title)
    ax.set_xlabel("Input x, y")
    if title.startswith("1"):
        ax.set_ylabel("Hidden Nodes (Sample 10)")

plot_heatmap(axes[0], old_weights, "1. Old Weights (Before Update)")
plot_heatmap(axes[1], gradients, "2. Gradients (By Backprop)")
plot_heatmap(axes[2], new_weights, "3. New Weights (After Adam Update)")

plt.tight_layout()
plt.show()


## 6. 업데이트 된 결과와 에포크 간 오차 비교

가중치가 1회 업데이트된 후, 다시 한 번 예측을 수행하여 Total Loss가 어떻게 변했는지 확인합니다.
PINN은 서로 다른 성격의 두 가지 Loss(MSE_u, MSE_f)가 결합되어 있어, 초반에는 두 Loss가 함께 줄어들거나 서로 상충(Trade-off)하는 현상을 관찰할 수도 있습니다.

In [ ]:
# ==========================================
## 6. 두 번째 순전파 및 오차 비교
# ==========================================

u_pred_bc_new, _ = model(X_u)
new_mse_u = nn.MSELoss()(u_pred_bc_new, target_bc_temp)
new_mse_f = calc_pde_loss_laplace(model, X_f)
new_total_loss = new_mse_u + new_mse_f

loss_epoch_1 = total_loss.item()
loss_epoch_2 = new_total_loss.item()

print(f"Epoch 1 전체 오차: {loss_epoch_1:.6f}")
print(f"Epoch 2 전체 오차: {loss_epoch_2:.6f}")
print(f"오차 변화량: {loss_epoch_1 - loss_epoch_2:.6f} 만큼 변화함.\n")

# 오차 비교 시각화 (막대 그래프)
fig, ax = plt.subplots(figsize=(6, 4))
epochs_labels = ['Epoch 1', 'Epoch 2']
losses = [loss_epoch_1, loss_epoch_2]

bars = ax.bar(epochs_labels, losses, color=['coral', 'lightblue'], width=0.5)

# 값 표기
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + (yval*0.01), f'{yval:.6f}', ha='center', va='bottom')

ax.set_ylabel('Total PINN Loss')
ax.set_title('Total Loss Comparison (Epoch 1 vs Epoch 2)')
ax.set_ylim(0, max(losses) * 1.2)
plt.show()


## 7. 본격적인 학습 1 : 라플라스 방정식 (Laplace Equation)

지금까지 원리를 파악했으니, 이제 설정한 파라미터(`epochs=5000`)를 통해 본격적인 PINN 학습 루프를 구동합니다.
**첫 번째로 풀 문제는 라플라스 방정식**입니다. 열이 평형 상태에 이르렀을 때의 온도 분포 등을 모델링할 때 쓰입니다.

**라플라스 방정식 문제 정의**:
*   **거버닝 방정식**: $\Delta u = u_{xx} + u_{yy} = 0$
*   **경계 조건**: 위쪽 경계($y=1$)만 $u(x,1)=\sin(\pi x)$ 의 열을 가지고, 나머지 3면의 경계는 $0$도로 유지합니다.
*   학습이 완료되면, 신경망 모델 자체가 주어진 경계조건과 물리법칙을 모두 만족하는 연속적인 수식 해(Solution)가 됩니다.

In [ ]:
# ==========================================
## 7. 반복 학습 수행 및 결과 시각화 (라플라스)
# ==========================================

# 깨끗한 학습을 위해 라플라스용 새 모델 초기화
model_laplace = DynamicPINN(input_size=2, hidden_layers=hidden_layers, output_size=1, activation=activation_func)
optimizer_laplace = optim.Adam(model_laplace.parameters(), lr=learning_rate)

# 라플라스 경계조건 정답(Target) 세팅
# X_u는 x=0, x=1, y=0, y=1 순서로 생성됨 (각각 pts_per_edge 개수만큼)
target_bc_laplace = torch.zeros(N_u, 1)
# y=1 인 구간(배열의 마지막 pts_per_edge 구간)에만 sin(pi*x) 적용
x_top_edge = X_u_x[-pts_per_edge:]
target_bc_laplace[-pts_per_edge:] = torch.sin(np.pi * x_top_edge)

loss_history_laplace = []
start_time = time.time()

print(f"--- 라플라스 방정식 총 {epochs} 에포크 학습 시작 ---")
for epoch in range(epochs):
    optimizer_laplace.zero_grad()
    
    # 1. 경계 조건 Loss (MSE_u)
    u_pred_bc, _ = model_laplace(X_u)
    mse_u = nn.MSELoss()(u_pred_bc, target_bc_laplace)
    
    # 2. 물리 법칙 Loss (MSE_f)
    mse_f = calc_pde_loss_laplace(model_laplace, X_f)
    
    # 3. Total Loss 및 역전파
    loss = mse_u + mse_f
    loss.backward()
    optimizer_laplace.step()
    
    loss_history_laplace.append(loss.item())
    
    # 진행률 10% 단위로 출력
    if (epoch + 1) % (epochs // 10) == 0 or (epoch + 1) == 1:
        print(f"Epoch [{epoch+1:4d}/{epochs}], Total Loss: {loss.item():.6e} (MSE_u: {mse_u.item():.6e}, MSE_f: {mse_f.item():.6e})")

print(f"\n라플라스 학습 완료! (소요 시간: {time.time() - start_time:.2f}초)")

# 결과 시각화 종합 패널 구성
fig = plt.figure(figsize=(15, 5))

# 1. Loss Curve (로그 스케일)
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(loss_history_laplace, color='blue', linewidth=2)
ax1.set_yscale('log') # PINN Loss는 폭넓게 떨어지므로 로그 스케일로 시각화
ax1.set_title('Laplace Training Loss Curve (Log Scale)')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Total PINN Loss')
ax1.grid(True, linestyle='--', alpha=0.6)

# 2. 예측된 해 u(x,y)의 2D Contour 분포 시각화
ax2 = fig.add_subplot(1, 2, 2)
# 평가를 위한 촘촘한 그리드 생성
x_grid = np.linspace(0, 1, 100)
y_grid = np.linspace(0, 1, 100)
X_mesh, Y_mesh = np.meshgrid(x_grid, y_grid)
grid_tensor = torch.FloatTensor(np.c_[X_mesh.ravel(), Y_mesh.ravel()])

with torch.no_grad():
    Z_pred, _ = model_laplace(grid_tensor)
Z_pred = Z_pred.numpy().reshape(X_mesh.shape)

contour = ax2.contourf(X_mesh, Y_mesh, Z_pred, levels=50, cmap='jet')
fig.colorbar(contour, ax=ax2)
ax2.set_title("Predicted Distribution u(x,y) - Laplace Eq")
ax2.set_xlabel("Input x")
ax2.set_ylabel("Input y")

plt.tight_layout()
plt.show()


## 8. 본격적인 학습 2 : 푸아송 방정식 (Poisson Equation)

라플라스 방정식과 유사하지만 내부에 발열체(소스, Source)가 존재하는 **푸아송 방정식**을 두 번째로 풀어봅니다.

**푸아송 방정식 문제 정의**:
*   **거버닝 방정식**: $\Delta u = f(x, y)$
*   **소스 함수**: $f(x, y) = -2\pi^2 \sin(\pi x) \sin(\pi y)$
*   **경계 조건**: 사각형 도메인의 모든 테두리에서 $u=0$ 을 유지합니다.
*   이 조건에서 수학적으로 계산된 완벽한 정답(Exact Solution)은 $u(x,y) = \sin(\pi x) \sin(\pi y)$ 입니다.

학습을 진행한 뒤, 딥러닝이 예측한 결과가 실제 수학적 정답과 얼마나 일치하는지 시각화하여 비교해 봅니다.

In [ ]:
# ==========================================
## 8. 반복 학습 수행 및 결과 시각화 (푸아송)
# ==========================================

# 푸아송용 모델 새롭게 초기화
model_poisson = DynamicPINN(input_size=2, hidden_layers=hidden_layers, output_size=1, activation=activation_func)
optimizer_poisson = optim.Adam(model_poisson.parameters(), lr=learning_rate)

# 푸아송 경계조건 정답 세팅: 모든 테두리에서 u=0
target_bc_poisson = torch.zeros(N_u, 1)

# 푸아송 방정식의 물리 잔차 함수 정의
def calc_pde_loss_poisson(model, x_f):
    u = model(x_f)[0]
    grad_u = torch.autograd.grad(u, x_f, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x = grad_u[:, 0:1]
    u_y = grad_u[:, 1:2]
    
    grad_u_x = torch.autograd.grad(u_x, x_f, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_xx = grad_u_x[:, 0:1]
    
    grad_u_y = torch.autograd.grad(u_y, x_f, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_yy = grad_u_y[:, 1:2]
    
    # 우변 함수 f(x,y)
    x = x_f[:, 0:1]
    y = x_f[:, 1:2]
    f_xy = -2 * (np.pi**2) * torch.sin(np.pi * x) * torch.sin(np.pi * y)
    
    # 푸아송 방정식 잔차: (u_xx + u_yy) - f(x,y) = 0
    residual = (u_xx + u_yy) - f_xy
    return torch.mean(residual ** 2)

loss_history_poisson = []
start_time = time.time()

print(f"--- 푸아송 방정식 총 {epochs} 에포크 학습 시작 ---")
for epoch in range(epochs):
    optimizer_poisson.zero_grad()
    
    u_pred_bc, _ = model_poisson(X_u)
    mse_u = nn.MSELoss()(u_pred_bc, target_bc_poisson)
    mse_f = calc_pde_loss_poisson(model_poisson, X_f)
    
    loss = mse_u + mse_f
    loss.backward()
    optimizer_poisson.step()
    
    loss_history_poisson.append(loss.item())
    
    if (epoch + 1) % (epochs // 10) == 0 or (epoch + 1) == 1:
        print(f"Epoch [{epoch+1:4d}/{epochs}], Total Loss: {loss.item():.6e}")

print(f"\n푸아송 학습 완료! (소요 시간: {time.time() - start_time:.2f}초)")

# 결과 시각화 (PINN 예측값 vs 실제 수학적 정답)
fig = plt.figure(figsize=(20, 5))

# 1. Loss Curve
ax1 = fig.add_subplot(1, 3, 1)
ax1.plot(loss_history_poisson, color='red', linewidth=2)
ax1.set_yscale('log')
ax1.set_title('Poisson Training Loss Curve')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Total PINN Loss')
ax1.grid(True, linestyle='--', alpha=0.6)

# 2. PINN 예측 모델의 결과
ax2 = fig.add_subplot(1, 3, 2)
with torch.no_grad():
    Z_pred_p, _ = model_poisson(grid_tensor)
Z_pred_p = Z_pred_p.numpy().reshape(X_mesh.shape)

contour_p = ax2.contourf(X_mesh, Y_mesh, Z_pred_p, levels=50, cmap='jet')
fig.colorbar(contour_p, ax=ax2)
ax2.set_title("PINN Prediction u(x,y)")
ax2.set_xlabel("x")
ax2.set_ylabel("y")

# 3. 실제 수학적 정답 (Exact Solution)
ax3 = fig.add_subplot(1, 3, 3)
exact_Z = np.sin(np.pi * X_mesh) * np.sin(np.pi * Y_mesh)

contour_e = ax3.contourf(X_mesh, Y_mesh, exact_Z, levels=50, cmap='jet')
fig.colorbar(contour_e, ax=ax3)
ax3.set_title("Exact Mathematical Solution u(x,y)")
ax3.set_xlabel("x")
ax3.set_ylabel("y")

plt.tight_layout()
plt.show()

# 오차율(L2 Relative Error) 계산 출력
error = np.linalg.norm(exact_Z - Z_pred_p) / np.linalg.norm(exact_Z)
print(f"\n🎯 정답 대비 예측 오차(L2 Relative Error): {error * 100:.3f}%")
